# Human Annotation Audit


## 0. Install/check packages


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "transformers": "transformers",
}
missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


## 1. Setup


In [ ]:
# Portable project-path setup.
# Override any of these by exporting the matching env var before launching
# the notebook (e.g. `export PROJECT_ROOT=/path/to/VulnerableCancerPatients`):
#   PROJECT_ROOT  - root of the experiment tree
#   SCRIPTS_DIR   - shared helper modules (default: PROJECT_ROOT/scripts)
#   DATA_DIR      - shared data directory (default: PROJECT_ROOT/data)
import os
import sys
from pathlib import Path

FOLDER_NAME = "07_HumanAnnotationAudit"
COLAB_DEFAULT = Path("/content/drive/MyDrive/NLP_Projects/VulnerableCancerPatients")


def _resolve_project_root() -> Path:
    env = os.environ.get("PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        if COLAB_DEFAULT.exists():
            return COLAB_DEFAULT.resolve()
    except ImportError:
        pass
    cwd = Path.cwd().resolve()
    if cwd.name == FOLDER_NAME:
        return cwd.parent
    if (cwd / FOLDER_NAME).exists():
        return cwd
    return cwd


PROJECT_ROOT = _resolve_project_root()
BASE_DIR = PROJECT_ROOT / FOLDER_NAME if (PROJECT_ROOT / FOLDER_NAME).exists() else PROJECT_ROOT
SCRIPTS_DIR = Path(os.environ.get("SCRIPTS_DIR", PROJECT_ROOT / "scripts")).expanduser().resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("Project root:", PROJECT_ROOT)
print("Base:", BASE_DIR)
print("Scripts:", SCRIPTS_DIR, "(exists)" if SCRIPTS_DIR.exists() else "(MISSING)")
print("Data:", DATA_DIR, "(exists)" if DATA_DIR.exists() else "(MISSING)")
print("Outputs:", OUTPUT_DIR)


## 2. Load shared data and split


In [ ]:

from data_utils import (
    EMOTION_LABELS_3,
    EMOTION_PROB_COLS_3,
    HEOR_SUBSCALES,
    HUMAN_HEOR_SUBSCALES,
    RAW_LLM_PROB_COLS_4,
    derive_high_need_flag,
    prepare_annotation_frame,
)
from metrics import (
    DEFAULT_BOOTSTRAP_N,
    bootstrap_classification_metric_rows,
    classification_metrics,
    multiclass_brier_score,
    one_hot,
    paired_delta_ci,
    soft_cross_entropy,
)

df = prepare_annotation_frame(ANNOTATION_PATH, SPLIT_PATH)
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Shape:", df.shape)
print(df["split"].value_counts().sort_index())
print("Bootstrap n:", DEFAULT_BOOTSTRAP_N)


## 3. Draw stratified human-validation sample


In [ ]:

SAMPLE_N = 240
RANDOM_STATE = 42
AI_EMOTION_4_LABELS = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE"]

# Stratify by LLM high-need prompt output, collapsed human emotion, and maximum
# HEOR level. This preserves enough severe/high-burden cases for the audit while
# retaining coverage across emotion strata.
sampling_frame = df.copy()
sampling_frame["heor_max_level"] = sampling_frame[HEOR_SUBSCALES].max(axis=1).astype(int)
sampling_frame["audit_stratum"] = (
    "high_need=" + sampling_frame["ai_high_need_flag"].astype(str)
    + "|emotion=" + sampling_frame["human_emotion_3class"].astype(str)
    + "|heor_max=" + sampling_frame["heor_max_level"].astype(str)
)

rng = np.random.default_rng(RANDOM_STATE)
groups = [(name, group.copy()) for name, group in sampling_frame.groupby("audit_stratum", sort=True)]
base_n = max(1, SAMPLE_N // len(groups))
selected_parts = []
selected_rows = set()

for _, group in groups:
    take = min(len(group), base_n)
    part = group.sample(take, random_state=RANDOM_STATE)
    selected_parts.append(part)
    selected_rows.update(part["source_row"].astype(int).tolist())

selected = pd.concat(selected_parts, ignore_index=True)
remaining_n = SAMPLE_N - len(selected)
if remaining_n > 0:
    rest = sampling_frame[~sampling_frame["source_row"].astype(int).isin(selected_rows)].copy()
    weights = 1.0 + rest["heor_max_level"].astype(float)
    weights = weights / weights.sum()
    extra_idx = rng.choice(rest.index.to_numpy(), size=min(remaining_n, len(rest)), replace=False, p=weights.to_numpy())
    selected = pd.concat([selected, rest.loc[extra_idx]], ignore_index=True)

audit_sample = selected.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
audit_sample["audit_id"] = [f"AUDIT_{i+1:04d}" for i in range(len(audit_sample))]
audit_sample["ai_emotion_4class"] = audit_sample[RAW_LLM_PROB_COLS_4].to_numpy().argmax(axis=1)
audit_sample["ai_emotion_4class"] = audit_sample["ai_emotion_4class"].map(dict(enumerate(AI_EMOTION_4_LABELS)))
audit_sample["ai_rule_derived_high_need_flag"] = derive_high_need_flag(
    audit_sample,
    emotion_col="ai_emotion_4class",
    benefit_col="ai_benefit_score",
    harm_col="ai_harm_score",
    cost_col="ai_cost_burden",
    treatment_col="ai_treatment_burden",
    life_col="ai_life_disruption",
    uncertainty_col="ai_uncertainty_conflict",
    support_col="ai_support_coping",
)

reference_cols = [
    "audit_id", "source_row", "posts", "audit_stratum", "human_emotion_3class",
    "llm_argmax_3class", "ai_emotion_4class", *RAW_LLM_PROB_COLS_4,
    "predicted", "intensity", "ai_speaker_role", "ai_cancer_type",
    "ai_high_need_flag", "ai_rule_derived_high_need_flag", *HEOR_SUBSCALES,
]
blinded_cols = ["audit_id", "source_row", "posts", "audit_stratum"]

audit_sample[reference_cols].to_csv(OUTPUT_DIR / "human_validation_sample_with_llm_reference.csv", index=False)
audit_sample[blinded_cols].to_csv(OUTPUT_DIR / "human_validation_sample_blinded.csv", index=False)
audit_sample[["audit_stratum"]].value_counts().reset_index(name="n").to_csv(
    OUTPUT_DIR / "human_validation_sample_strata.csv", index=False
)

audit_sample[["audit_id", "source_row", "audit_stratum"]].head()


## 4. Export coder rubric


In [ ]:

LEVEL_ANCHORS = (
    "0 = absent / not expressed in the post; "
    "1 = mild or brief mention; "
    "2 = moderate, explicit, substantive expression; "
    "3 = severe, pervasive, central, or overwhelming expression"
)
HIGH_NEED_RULE = (
    "Mark TRUE if the post meets any of the high-need criteria: composite well-being score <=25, "
    "emotion_4class is VERY_NEGATIVE, harm_score = 3, or uncertainty_conflict = 3. "
    "Composite score = 50 + 10*benefit_score + 6*support_coping - 10*harm_score "
    "- 8*cost_burden - 8*treatment_burden - 8*life_disruption - 8*uncertainty_conflict, "
    "clamped to 0-100."
)
DERIVED_HIGH_NEED_RULE = (
    "Formula-derived from emotion_4class and the 7 subscale ratings using the same rule as high_need_flag. "
    "This field is used to compare human-derived vs LLM-derived flags separately from coder-entered high_need_flag."
)

rubric = pd.DataFrame([
    {"field": "speaker_role", "allowed_values": "PATIENT; CAREGIVER; UNCLEAR", "guidance": "Label the author perspective expressed in the post. Use UNCLEAR if not inferable.", "level_anchors": ""},
    {"field": "cancer_type", "allowed_values": "BRAIN; COLON; LIVER; LEUKEMIA; LUNG; OTHER; UNKNOWN", "guidance": "Use a cancer type only if explicit or unambiguous. UNKNOWN if cancer is mentioned without type. OTHER for explicit types outside the listed cancer types. Examples: BRAIN = glioblastoma/GBM/astrocytoma/brain tumor; COLON = colon cancer/colorectal/CRC/rectal cancer; LIVER = HCC/hepatocellular/liver cancer; LEUKEMIA = AML/ALL/CML/CLL/leukemia; LUNG = NSCLC/SCLC/lung cancer.", "level_anchors": ""},
    {"field": "emotion_4class", "allowed_values": "VERY_NEGATIVE; NEGATIVE; NEUTRAL; POSITIVE", "guidance": "Code the dominant overall emotional tone of the post. Keep VERY_NEGATIVE for clear dominant severe negative tone, such as acute fear, despair, overwhelming distress, crisis language, or intense grief. Use NEGATIVE for worry, sadness, frustration, or negative affect that is present but less severe or not dominant.", "level_anchors": "Ordinal mapping for IRR: 0 = VERY_NEGATIVE; 1 = NEGATIVE; 2 = NEUTRAL; 3 = POSITIVE."},
    {"field": "benefit_score", "allowed_values": "0; 1; 2; 3", "guidance": "Perceived treatment benefit or positive care outcome.", "level_anchors": LEVEL_ANCHORS},
    {"field": "harm_score", "allowed_values": "0; 1; 2; 3", "guidance": "Perceived harm, treatment toxicity, side effects, adverse events, or harm-from-care narrative.", "level_anchors": LEVEL_ANCHORS},
    {"field": "cost_burden", "allowed_values": "0; 1; 2; 3", "guidance": "Financial toxicity, insurance, affordability, transportation cost, lost income, or cost-related access burden.", "level_anchors": LEVEL_ANCHORS},
    {"field": "treatment_burden", "allowed_values": "0; 1; 2; 3", "guidance": "Logistical, regimen, appointment, medication, monitoring, or care-process burden.", "level_anchors": LEVEL_ANCHORS},
    {"field": "life_disruption", "allowed_values": "0; 1; 2; 3", "guidance": "Disruption to work, family, daily functioning, identity, plans, independence, or quality of life.", "level_anchors": LEVEL_ANCHORS},
    {"field": "uncertainty_conflict", "allowed_values": "0; 1; 2; 3", "guidance": "Prognostic uncertainty, decisional conflict, confusion, waiting, insufficient information, or not knowing what to do.", "level_anchors": LEVEL_ANCHORS},
    {"field": "support_coping", "allowed_values": "0; 1; 2; 3", "guidance": "Support, coping resources, reassurance, acceptance, practical help, peer support, family support, or clinician support. This is protective, not burden.", "level_anchors": LEVEL_ANCHORS},
    {"field": "high_need_flag", "allowed_values": "TRUE; FALSE", "guidance": HIGH_NEED_RULE, "level_anchors": "Coder-entered summary flag. Analysis also derives a rule-based flag from the coded emotion and subscale fields."},
    {"field": "derived_high_need_flag", "allowed_values": "derived TRUE/FALSE", "guidance": DERIVED_HIGH_NEED_RULE, "level_anchors": "Coders do not edit this value manually."},
    {"field": "evidence", "allowed_values": "free text", "guidance": "Optional short phrase from the post supporting difficult ratings. Do not paste long excerpts.", "level_anchors": ""},
    {"field": "notes", "allowed_values": "free text", "guidance": "Optional coder notes or uncertainty flags.", "level_anchors": ""},
])
rubric.to_csv(OUTPUT_DIR / "human_annotation_rubric.csv", index=False)
rubric


## 5. Export coder template


In [ ]:

CODER_COLUMNS = [
    "audit_id", "source_row", "posts", "coder_id", "speaker_role", "cancer_type", "emotion_4class",
    *HUMAN_HEOR_SUBSCALES, "high_need_flag", "derived_high_need_flag", "evidence", "notes",
]
AI_VISIBLE_REVIEW_COLUMNS = [
    "audit_id", "source_row", "posts",
    "ai_speaker_role", "reviewer_speaker_role",
    "ai_cancer_type", "reviewer_cancer_type",
    "ai_emotion_4class", "reviewer_emotion_4class",
    "ai_benefit_score", "reviewer_benefit_score",
    "ai_harm_score", "reviewer_harm_score",
    "ai_cost_burden", "reviewer_cost_burden",
    "ai_treatment_burden", "reviewer_treatment_burden",
    "ai_life_disruption", "reviewer_life_disruption",
    "ai_uncertainty_conflict", "reviewer_uncertainty_conflict",
    "ai_support_coping", "reviewer_support_coping",
    "ai_high_need_flag", "reviewer_high_need_flag",
    "ai_rule_derived_high_need_flag", "reviewer_derived_high_need_flag",
    "evidence", "notes",
]


def make_coder_template(sample, coder_id):
    template = sample[["audit_id", "source_row", "posts"]].copy()
    template["coder_id"] = coder_id
    for col in ["speaker_role", "cancer_type", "emotion_4class", *HUMAN_HEOR_SUBSCALES, "high_need_flag"]:
        template[col] = ""
    template["derived_high_need_flag"] = ""
    template["evidence"] = ""
    template["notes"] = ""
    return template[CODER_COLUMNS]


def make_ai_visible_review_template(sample):
    template = sample[["audit_id", "source_row", "posts"]].copy()
    template["ai_speaker_role"] = sample["ai_speaker_role"].fillna("UNCLEAR").astype(str).str.upper()
    template["reviewer_speaker_role"] = ""
    template["ai_cancer_type"] = sample["ai_cancer_type"].fillna("UNKNOWN").astype(str).str.upper()
    template["reviewer_cancer_type"] = ""
    template["ai_emotion_4class"] = sample["ai_emotion_4class"]
    template["reviewer_emotion_4class"] = ""
    for ai_col, reviewer_col in [
        ("ai_benefit_score", "reviewer_benefit_score"),
        ("ai_harm_score", "reviewer_harm_score"),
        ("ai_cost_burden", "reviewer_cost_burden"),
        ("ai_treatment_burden", "reviewer_treatment_burden"),
        ("ai_life_disruption", "reviewer_life_disruption"),
        ("ai_uncertainty_conflict", "reviewer_uncertainty_conflict"),
        ("ai_support_coping", "reviewer_support_coping"),
    ]:
        template[ai_col] = sample[ai_col].astype(int)
        template[reviewer_col] = ""
    template["ai_high_need_flag"] = sample["ai_high_need_flag"].map({1: "TRUE", 0: "FALSE"}).fillna("FALSE")
    template["reviewer_high_need_flag"] = ""
    template["ai_rule_derived_high_need_flag"] = sample["ai_rule_derived_high_need_flag"].map({1: "TRUE", 0: "FALSE"})
    template["reviewer_derived_high_need_flag"] = ""
    template["evidence"] = ""
    template["notes"] = ""
    return template[AI_VISIBLE_REVIEW_COLUMNS]


coder1 = make_coder_template(audit_sample, "CODER_1")
coder2 = make_coder_template(audit_sample, "CODER_2")
ai_visible_review = make_ai_visible_review_template(audit_sample)

coder1.to_csv(OUTPUT_DIR / "human_annotation_coder_1.csv", index=False)
coder2.to_csv(OUTPUT_DIR / "human_annotation_coder_2.csv", index=False)
ai_visible_review.to_csv(OUTPUT_DIR / "human_annotation_ai_visible_review.csv", index=False)
pd.concat([coder1, coder2], ignore_index=True).to_csv(
    OUTPUT_DIR / "human_annotation_completed_long_template.csv", index=False
)

coder1.head()


## 6. Inter-rater reliability after coding


In [ ]:

from sklearn.metrics import cohen_kappa_score, confusion_matrix

EMOTION_4CLASS_TO_ORDINAL = {"VERY_NEGATIVE": 0, "NEGATIVE": 1, "NEUTRAL": 2, "POSITIVE": 3}
FLAG_FIELDS = {"high_need_flag", "derived_high_need_flag"}


def bootstrap_kappa_ci(labels_a, labels_b, weights=None, n_boot=DEFAULT_BOOTSTRAP_N, seed=42):
    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    observed = float(cohen_kappa_score(labels_a, labels_b, weights=weights))
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(labels_a), len(labels_a))
        try:
            samples.append(float(cohen_kappa_score(labels_a[idx], labels_b[idx], weights=weights)))
        except Exception:
            samples.append(np.nan)
    samples = np.asarray(samples, dtype=float)
    samples = samples[np.isfinite(samples)]
    if len(samples) == 0:
        return observed, np.nan, np.nan
    return observed, float(np.percentile(samples, 2.5)), float(np.percentile(samples, 97.5))


def normalize_flag(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .replace({"1": "TRUE", "1.0": "TRUE", "YES": "TRUE", "Y": "TRUE", "0": "FALSE", "0.0": "FALSE", "NO": "FALSE", "N": "FALSE"})
    )


def add_human_derived_high_need(frame):
    frame = frame.copy()
    required = {"emotion_4class", *HUMAN_HEOR_SUBSCALES}
    if required <= set(frame.columns):
        frame["derived_high_need_flag"] = derive_high_need_flag(
            frame,
            emotion_col="emotion_4class",
            benefit_col="benefit_score",
            harm_col="harm_score",
            cost_col="cost_burden",
            treatment_col="treatment_burden",
            life_col="life_disruption",
            uncertainty_col="uncertainty_conflict",
            support_col="support_coping",
        ).map({1: "TRUE", 0: "FALSE"})
    return frame


def kappa_inputs(paired, col):
    a = paired[f"{col}_a"].copy()
    b = paired[f"{col}_b"].copy()
    if col == "emotion_4class":
        a = a.astype(str).str.upper().str.replace(" ", "_", regex=False).map(EMOTION_4CLASS_TO_ORDINAL)
        b = b.astype(str).str.upper().str.replace(" ", "_", regex=False).map(EMOTION_4CLASS_TO_ORDINAL)
        keep = a.notna() & b.notna()
        return a[keep].astype(int), b[keep].astype(int), "quadratic"
    if col in HUMAN_HEOR_SUBSCALES:
        a = pd.to_numeric(a, errors="coerce")
        b = pd.to_numeric(b, errors="coerce")
        keep = a.notna() & b.notna()
        return a[keep].astype(int), b[keep].astype(int), "quadratic"
    if col in FLAG_FIELDS:
        a = normalize_flag(a)
        b = normalize_flag(b)
        keep = a.isin(["TRUE", "FALSE"]) & b.isin(["TRUE", "FALSE"])
        return a[keep], b[keep], None
    keep = a.notna() & b.notna()
    return a[keep].astype(str), b[keep].astype(str), None


CODER_FILE = OUTPUT_DIR / "human_annotation_completed.csv"

if not CODER_FILE.exists():
    print(f"Place completed human annotations at: {CODER_FILE}")
else:
    human = pd.read_csv(CODER_FILE)
    human = add_human_derived_high_need(human)
    required = {"audit_id", "coder_id"}
    missing = required - set(human.columns)
    if missing:
        raise KeyError(f"Completed coder file missing columns: {sorted(missing)}")

    label_cols = [
        "speaker_role",
        "cancer_type",
        "emotion_4class",
        *HUMAN_HEOR_SUBSCALES,
        "high_need_flag",
        "derived_high_need_flag",
    ]
    coder_ids = sorted(human["coder_id"].dropna().unique())
    if len(coder_ids) < 2:
        raise ValueError("Need at least two coder_id values for inter-rater reliability.")

    wide = {}
    for coder in coder_ids[:2]:
        wide[coder] = human[human["coder_id"] == coder].set_index("audit_id")

    irr_rows = []
    for col in label_cols:
        if col not in human.columns:
            continue
        paired = wide[coder_ids[0]][[col]].join(wide[coder_ids[1]][[col]], lsuffix="_a", rsuffix="_b").dropna()
        if len(paired) == 0:
            continue
        labels_a, labels_b, weights = kappa_inputs(paired, col)
        if len(labels_a) == 0:
            continue
        kappa, lo, hi = bootstrap_kappa_ci(labels_a, labels_b, weights=weights)
        irr_rows.append({
            "field": col,
            "coder_a": coder_ids[0],
            "coder_b": coder_ids[1],
            "n": len(labels_a),
            "kappa": kappa,
            "kappa_ci_low": lo,
            "kappa_ci_high": hi,
            "n_boot": DEFAULT_BOOTSTRAP_N,
            "ci_method": "nonparametric_percentile",
            "weights": weights or "none",
        })

    irr = pd.DataFrame(irr_rows)
    irr.to_csv(OUTPUT_DIR / "human_interrater_reliability.csv", index=False)
    irr


## 7. Human-vs-LLM agreement after consensus


In [ ]:

CODER_CONSENSUS_FILE = OUTPUT_DIR / "human_annotation_consensus.csv"

if not CODER_CONSENSUS_FILE.exists():
    print(f"Place consensus human annotations at: {CODER_CONSENSUS_FILE}")
else:
    consensus = pd.read_csv(CODER_CONSENSUS_FILE)
    consensus = add_human_derived_high_need(consensus)
    merged = consensus.merge(audit_sample, on=["audit_id", "source_row"], suffixes=("_human", "_llm"), validate="one_to_one")

    comparison_rows = []
    comparison_specs = [
        ("speaker_role", "ai_speaker_role", "speaker_role"),
        ("cancer_type", "ai_cancer_type", "cancer_type"),
        ("emotion_4class", "ai_emotion_4class", "emotion_4class"),
        ("high_need_flag", "ai_high_need_flag", "high_need_flag_human_entered_vs_llm_prompt"),
        ("derived_high_need_flag", "ai_rule_derived_high_need_flag", "high_need_flag_human_derived_vs_llm_derived"),
        ("benefit_score", "ai_benefit_score", "benefit_score"),
        ("harm_score", "ai_harm_score", "harm_score"),
        ("cost_burden", "ai_cost_burden", "cost_burden"),
        ("treatment_burden", "ai_treatment_burden", "treatment_burden"),
        ("life_disruption", "ai_life_disruption", "life_disruption"),
        ("uncertainty_conflict", "ai_uncertainty_conflict", "uncertainty_conflict"),
        ("support_coping", "ai_support_coping", "support_coping"),
    ]

    for human_col, llm_col, output_field in comparison_specs:
        human_merged_col = human_col if human_col in merged.columns else f"{human_col}_human"
        llm_merged_col = llm_col if llm_col in merged.columns else f"{llm_col}_llm"
        if human_merged_col not in merged.columns or llm_merged_col not in merged.columns:
            continue
        temp = merged[[human_merged_col, llm_merged_col]].dropna().rename(
            columns={human_merged_col: f"{human_col}_a", llm_merged_col: f"{human_col}_b"}
        )
        if len(temp) == 0:
            continue
        labels_a, labels_b, weights = kappa_inputs(temp, human_col)
        if len(labels_a) == 0:
            continue
        kappa, lo, hi = bootstrap_kappa_ci(labels_a, labels_b, weights=weights)
        comparison_rows.append({
            "field": output_field,
            "human_field": human_col,
            "llm_field": llm_col,
            "n": len(labels_a),
            "kappa": kappa,
            "kappa_ci_low": lo,
            "kappa_ci_high": hi,
            "n_boot": DEFAULT_BOOTSTRAP_N,
            "ci_method": "nonparametric_percentile",
            "weights": weights or "none",
        })
        pd.crosstab(pd.Series(labels_a, name="human"), pd.Series(labels_b, name="llm")).to_csv(
            OUTPUT_DIR / f"human_vs_llm_confusion_{output_field}.csv"
        )

    agreement = pd.DataFrame(comparison_rows)
    agreement.to_csv(OUTPUT_DIR / "human_llm_agreement_metrics.csv", index=False)
    agreement
